In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

mlp_term_3_2025_kaggle_assignment_1_path = kagglehub.competition_download('mlp-term-3-2025-kaggle-assignment-1')

print('Data source import complete.')


# Importing the necessary packages

In [ ]:
# Data Analysis and Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Train-Test Splitting and Hyper-parameter Tuning
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from scipy.stats import uniform, loguniform, randint

# Models
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, AdaBoostRegressor, VotingRegressor, StackingRegressor
from sklearn.neural_network import MLPRegressor

# Preformance Metrics
from sklearn.metrics import r2_score

# Getting rid of Warnings
import warnings
warnings.filterwarnings("ignore")

# Setting Configs
%matplotlib inline

# Data Reading

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Reading the train and test datasets
train_df = pd.read_csv("/kaggle/input/mlp-term-3-2025-kaggle-assignment-1/train.csv")
test_df = pd.read_csv("/kaggle/input/mlp-term-3-2025-kaggle-assignment-1/test.csv")

In [ ]:
train_df.head()

# Creating a Baseline Model (Dummy Regressor)

In [ ]:
# Seperating the feature and target columns
X = train_df.drop("price", axis=1)
y = train_df["price"]

# Initializing the dummy regressor and fitting it
model = DummyRegressor().fit(X, y)

# Testing the baseline model
X_test = test_df
y_pred = model.predict(X_test)

# Submission file generation
submission = pd.DataFrame({
    "id": range(0, X_test.shape[0]),
    "target": y_pred
})
submission.to_csv("submission.csv", index=False)

# Taking a look on the training dataset

In [ ]:
# Shape of the training set
train_df.shape

The training dataset has 1000 rows, 1 ID column, 7 input features and 1 target (price) column.

# Having a look at randomly sampled datapoints from the dataset.

In [ ]:
# .sample(no of samples, default=1) returns some randomly selected datapoints from the dataset at each execution
train_df.sample(7)

# Gathering the information about the various features in the training set.

In [ ]:
train_df.info()

There are **5 numerical** (1 integer and 4 float) features in our training set, remaining **4 features are categorical**.

# Descriptive statistics of the numerical features of the dataset

In [ ]:
train_df.describe().T # transpose of the descriptive statistics matrix

# Look for any duplicated rows in the dataset.

Since `id` is an unique feature for each row, for checking the duplicates, we need to drop the `id` feature and then look for the number of duplicates in our training set.

In [ ]:
# Dropping id column
train_df_without_id = train_df.drop("id", axis=1)
train_df_without_id.duplicated().sum()

In [ ]:
# Droping the duplicated rows
train_df_cleaned = train_df_without_id.drop_duplicates()
train_df_cleaned.shape

This shows that all the rows in our dataset are not unique, there are 361 rows that repeat. Hence we have dropped the duplicated rows.

# Looking at the correlation among various features of the training set

In [ ]:
numerical_cols = ["total_sqft", "bath", "balcony", "price"]
corr = train_df_cleaned[numerical_cols].corr() # .corr function returns the correlation matrix
corr

We can observe from all of the cells, that there is a **positive correlation** value, with `total_sqft` and `price` having the strongest positive correlation.

# Exploratory Data Analysis

As part of EDA, we are supposed to identify and infer some trends and information from the dataset given to us.

There are different types of EDA we can perform. They are:

- Univerate EDA
- Bivariate EDA
- Multivariate EDA

In [ ]:
# Seperating the numerical and categorical features
num_features = train_df_cleaned.select_dtypes(include=["float64", "int64"]).columns
cat_features = train_df_cleaned.select_dtypes(include=["object"]).columns

print(f"Numerical Features: {num_features}")
print(f"Categorical Features: {cat_features}")

In [ ]:
# Observing the count of various categories in the training set
for i in train_df_cleaned:
    print(train_df_cleaned[i].value_counts())
    print("-"*50)

In [ ]:
# Unique values per column
for i in train_df_cleaned:
    unique = train_df_cleaned[i].unique()
    dtype = train_df_cleaned[i].dtype
    print(f"{i} ({dtype}): {len(unique)} unique values", f"\n=> {unique}\n" if len(unique) < 50 else "\n")

Following are the inferences (include missing values) which we obtain from the above operations:

1. `id`: There is a unique entry in each of the rows for the identity feature.
2. `area_type`: There are **4** unique categories in this feature. The one with the maximum count is **type_I**.
3. `availability`: There are **76** unique categories in this feature. The one with the maximum count is **Ready To Move**.
4. `location`: There are **1186** unique categories in this feature. The one with the maximum count is **Whitefield**.
5. `size`: There are **28** unique categories in this feature. The one with the maximum count is **2 BHK**.
6. `total_sqft`: There are **1801** unique categories in this feature. The one with the maximum count is **1200.0**.
7. `bath`: There are **17** unique categories in this feature. The one with the maximum count is **2.0**.
8. `balcony`: There are **5** unique categories in this feature. The one with the maximum count is **2.0**.
9. `price`: There are **1695** unique categories in this feature. The one with the maximum count is **75.00**.

# Cleaning the dataset

## Handling `size` column

Since the `size` column has multiple representations for similar sizes (ex. '3 BHK' and '3 Beedroom'), it needs to be converted into a numerical column using the `apply()` method.

In [ ]:
def extract_bhk_number(size_str):
    # First, check if the value is already missing (NaN)
    if pd.isnull(size_str):
        return np.nan

    parts = str(size_str).split(" ")
    return float(parts[0])

train_df_cleaned["size"] = train_df_cleaned["size"].apply(extract_bhk_number)

In [ ]:
train_df_cleaned["size"].unique()

We can now notice that the `size` column is a numeric column.

## Handling `availability` column

We also need to clean the `availability` column since the column has 76 unique values and is of categorical type. So one-hot encoding would lead to 76 new columns. Instead we will convert it to a binary column where `1` means it is Ready to Move and `0` means it is available sometime in the future.

In [ ]:
ready_to_move_mask = train_df_cleaned["availability"] == "Ready To Move"
# Convert "availability" to a binary column
# Where value is "Ready To Move", set to 1, otherwise set to 0
train_df_cleaned["availability"] = np.where(ready_to_move_mask, 1, 0)

In [ ]:
train_df_cleaned["availability"].value_counts()

We can now notice that the `availability` column is a binary column with **7663** 1's and **1976** 0's.

## Handling `location` column

We also need to clean the `location` column since the column has 1186 unique values and is of categorical type. So one-hot encoding would lead to 1186 new columns. Instead we will group all locations which appear less than 10 times in "Other" location. Rest of the columns can be one-hot encoded.

In [ ]:
# Get value counts
location_stats = train_df_cleaned["location"].value_counts()

# Identify locations to group (those with 10 or fewer entries)
locations_to_group = location_stats[location_stats <= 10].index

# Use .apply() with a lambda function.
# If a location is in our "locations_to_group" list, replace it with "Other".
# Otherwise, keep the location name.
train_df_cleaned["location"] = train_df_cleaned["location"].apply(
    lambda x: "Other" if x in locations_to_group else x
)

In [ ]:
train_df_cleaned["location"].nunique()

In [ ]:
train_df_cleaned["location"].value_counts()

We can see that we have reduced the number of unique values in `location` from 1186 down to 188. Reducing any further will lead to data loss. Hence we will now apply one-hot encoding on this later.

## Looking at updated `.info()`

In [ ]:
train_df_cleaned.info()

# Look for the missing values in the various features of the training set.

In [ ]:
# Number of missing values in each feature of the training set.
train_df_cleaned.isna().sum().sort_values(ascending=False)

Executing the above command shows that the following column has missing values:

- balcony
- bath
- total_sqft
- size
- location

In [ ]:
# Total number of missing values in the dataset
train_df_cleaned.isna().sum().sum()

The total number of missing values in the training set are 581.

## Imputing the missing values

We will use `SimpleImputer` for imputing. We will use `median` value for `total_sqft`, and `mode` for the others (since they are discrete/categorical).

In [ ]:
# Imputing total_sqft
median_imputer = SimpleImputer(strategy="median")
train_df_cleaned["total_sqft"] = median_imputer.fit_transform(train_df_cleaned[["total_sqft"]])

# Imputing discrete/categorical columns
mode_imputer = SimpleImputer(strategy="most_frequent")
mode_cols = ["balcony", "bath", "size"]
train_df_cleaned[mode_cols] = mode_imputer.fit_transform(train_df_cleaned[mode_cols])
train_df_cleaned["location"] = mode_imputer.fit_transform(train_df_cleaned[["location"]]).ravel()

In [ ]:
# Missing values after imputation
train_df_cleaned.isna().sum()

We can see that there are no missing values left after imputation.

In [ ]:
train_df_cleaned.info()

# Detecting the Outliers

We will use a domain-specific approach, which involves the following 2 steps:

1. **Removing physical impossibilities:** We'll remove houses where the number of bathrooms is unreasonably high compared to the number of bedrooms.
2. **Removing price outliers:** We'll calculate "price per square foot" and remove the extreme high and low values, which are almost always data errors (e.g., a house listed for $\$1/sqft$ or $\$1,000,000/sqft$).

In [ ]:
# Shape before handling outliers
train_df_cleaned.shape

In [ ]:
bath_outliers = train_df_cleaned[train_df_cleaned["bath"] > train_df_cleaned["size"] + 2]
len(bath_outliers)

These are 10 outliers where there are way more bathrooms that the number of bedrooms. We are dropping these.

In [ ]:
train_df_cleaned = train_df_cleaned[train_df_cleaned["bath"] < train_df_cleaned["size"] + 2]

In [ ]:
# Price is in Lakhs, so we multiply by 100,000
train_df_cleaned["price_per_sqft"] = (train_df_cleaned["price"] * 100000) / train_df_cleaned["total_sqft"]

In [ ]:
# We'll use the 2th and 98th percentiles as our cutoffs
# This removes the most extreme 4% of the data
lower_limit = train_df_cleaned["price_per_sqft"].quantile(0.02)
upper_limit = train_df_cleaned["price_per_sqft"].quantile(0.98)
print(f"'price_per_sqft' lower limit (2th percentile): {lower_limit:.2f}")
print(f"'price_per_sqft' upper limit (98th percentile): {upper_limit:.2f}")

In [ ]:
price_per_sqft_mask = (train_df_cleaned["price_per_sqft"] >= lower_limit) & \
                      (train_df_cleaned["price_per_sqft"] <= upper_limit)
train_df_cleaned = train_df_cleaned[price_per_sqft_mask]

In [ ]:
train_df_cleaned = train_df_cleaned.drop("price_per_sqft", axis=1)

In [ ]:
# Shape after handling outliers
train_df_cleaned.shape

In [ ]:
train_df_cleaned.info()

After handling outliers, we are left with 9146 rows in our dataset.

# Visualizing data

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(train_df_cleaned["price"], kde=True, bins=50)
plt.title("Distribution of House Price")
plt.xlabel("Price (in Lakhs)")
plt.ylabel("Frequency")

plt.subplot(1, 2, 2)
sns.histplot(np.log1p(train_df_cleaned["price"]), kde=True, bins=50, color="green")
plt.title("Distribution of Log-Transformed Price")
plt.xlabel("Log(Price + 1)")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

**Insight:** The price is heavily right-skewed. Log-transforming it (right plot) makes it much more 'normal', which is better for most regression models.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x="total_sqft", y="price", data=train_df_cleaned, alpha=0.5)
plt.title("Price vs. Total Square Feet")
plt.xlabel("Total Sqft")
plt.ylabel("Price (in Lakhs)")
plt.show()

**Insight:** There's a strong positive correlation; as square footage increases, the price generally increases.

In [ ]:
plt.figure(figsize=(10, 6))
# Plot for sizes < 8 BHK, as larger ones are rare outliers
sns.boxplot(x="size", y="price", data=train_df_cleaned[train_df_cleaned["size"] < 8])
plt.title("Price vs. Size (BHK)")
plt.xlabel("Number of Bedrooms (Size)")
plt.ylabel("Price (in Lakhs)")
plt.show()

**Insight:** The median price (the line in the box) and the price range (the box height) both clearly increase with the number of bedrooms.

In [ ]:
plt.figure(figsize=(12, 7))
# Find top 10 most frequent locations (excluding "Other")
top_10_locations = train_df_cleaned[train_df_cleaned["location"] != "Other"]["location"]\
                   .value_counts().head(10).index
df_top10_loc = train_df_cleaned[train_df_cleaned["location"].isin(top_10_locations)]

# Calculate mean price and sort
location_prices = df_top10_loc.groupby("location")["price"].mean().sort_values(ascending=False)

sns.barplot(x=location_prices.values, y=location_prices.index, orient="h")
plt.title("Mean Price for Top 10 Most Frequent Locations")
plt.xlabel("Mean Price (in Lakhs)")
plt.ylabel("Location")
plt.tight_layout()
plt.show()

**Insight:** Location is a huge price driver. The mean price varies significantly even just between the 10 most common locations.

In [ ]:
plt.figure()
numeric_cols = ["price", "total_sqft", "size", "bath", "balcony"]
sns.heatmap(train_df_cleaned[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap of Numeric Features")
plt.tight_layout()
plt.show()

**Insight:** `price` is most strongly correlated with `total_sqft` ($0.81$) and `bath` ($0.50$). Also, `size` and `bath` are highly correlated ($0.89$), which makes sense.

# Encoding & Scaling

- `OneHotEncoder()` for `area_type` and `location`.
- `StandardScaler()` for `total_sqft`, `size`, `bath` and `balcony`.
- `price` and `availability` will be left as is.

In [ ]:
numeric_features = ["total_sqft", "size", "bath", "balcony"]
categorical_features = ["area_type", "location"]
binary_feature = ["availability"]

# Numeric Pipeline
num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
]).set_output(transform="pandas")

# Categorical Pipeline
cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
]).set_output(transform="pandas")

pipeline = ColumnTransformer(
    transformers=[
        ("num", num_pipe, numeric_features),
        ("cat", cat_pipe, categorical_features),
        ("binary", "passthrough", binary_feature)
    ],
    remainder="drop"
).set_output(transform="pandas")

In [ ]:
# Visualizing the pipeline
pipeline

# Separating the training and validation set & data transformation

In [ ]:
X = train_df_cleaned.drop("price", axis=1)
y = train_df_cleaned["price"]

In [ ]:
# Seperating the train and validation set
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_val.shape, y_train.shape, y_val.shape

In [ ]:
# Train and validation set transformation
X_train_transformed = pipeline.fit_transform(X_train)
X_val_transformed = pipeline.transform(X_val)

In [ ]:
X_train_transformed.head()

# Model training and perfomance analysis

## Model fitting

In [ ]:
# 1. Default Linear Regression
lr = LinearRegression()
lr.fit(X_train_transformed, y_train)

In [ ]:
# 2. Default SGD Regressor
sgdr = SGDRegressor(random_state=42)
sgdr.fit(X_train_transformed, y_train)

In [ ]:
# 3. Default Lasso Regressor
lasso = Lasso(random_state=42)
lasso.fit(X_train_transformed, y_train)

In [ ]:
# 4. Default Ridge Regressor
ridge = Ridge(random_state=42)
ridge.fit(X_train_transformed, y_train)

In [ ]:
# 5. K Neighbors Regressor with neighbors = 5
KNreg = KNeighborsRegressor(n_neighbors=5)
KNreg.fit(X_train_transformed, y_train)

In [ ]:
# 6. Default Support Vector Regressor
svr = SVR()
svr.fit(X_train_transformed, y_train)

In [ ]:
# 7. Default Decision Tree Regression
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train_transformed, y_train)

In [ ]:
# 8. Default Bagging Regressor
bag = BaggingRegressor(random_state=42)
bag.fit(X_train_transformed, y_train)

In [ ]:
# 9. Default Adaptive-Boosting Regressor
adaboost = AdaBoostRegressor(random_state=42)
adaboost.fit(X_train_transformed, y_train)

In [ ]:
# 10. Default Random Forest Regressor
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train_transformed, y_train)

In [ ]:
# 11. Default Multi-Layer Perceptron Regressor
mlpreg = MLPRegressor(random_state=42, early_stopping=True)
mlpreg.fit(X_train_transformed, y_train)

## Model performance evaluation

In [ ]:
lr_r2 = r2_score(y_val, lr.predict(X_val_transformed))
lr_r2

In [ ]:
sgdr_r2 = r2_score(y_val, sgdr.predict(X_val_transformed))
sgdr_r2

In [ ]:
lasso_r2 = r2_score(y_val, lasso.predict(X_val_transformed))
lasso_r2

In [ ]:
ridge_r2 = r2_score(y_val, ridge.predict(X_val_transformed))
ridge_r2

In [ ]:
KNreg_r2 = r2_score(y_val, KNreg.predict(X_val_transformed))
KNreg_r2

In [ ]:
svr_r2 = r2_score(y_val, svr.predict(X_val_transformed))
svr_r2

In [ ]:
dt_r2 = r2_score(y_val, dt.predict(X_val_transformed))
dt_r2

In [ ]:
bag_r2 = r2_score(y_val, bag.predict(X_val_transformed))
bag_r2

In [ ]:
adaboost_r2 = r2_score(y_val, adaboost.predict(X_val_transformed))
adaboost_r2

In [ ]:
rf_r2 = r2_score(y_val, rf.predict(X_val_transformed))
rf_r2

In [ ]:
mlpreg_r2 = r2_score(y_val, mlpreg.predict(X_val_transformed))
mlpreg_r2

In [ ]:
# r2_score Camparision Bar Plot
models = ["LinReg", "SGDReg", "Lasso", "Ridge", "K-NeighReg", "DeciTree", "BaggReg", "AdaBoostReg", "RanForestReg", "MLPReg"]
scores = [lr_r2, sgdr_r2, lasso_r2, ridge_r2, KNreg_r2, dt_r2, bag_r2, adaboost_r2, rf_r2, mlpreg_r2]

sns.barplot(x=models, y=scores, palette="cividis")
plt.xlabel("Models")
plt.xticks(rotation=45, ha="right")
plt.ylabel("R2-Score")
plt.title("Model-wise R2-Scores on Validation Set")
plt.ylim(0.45, 0.80)
plt.show()

From the above graph, we can infer that the models that give best performance (r2-score) on validation set are:
1. K-Neighbors Regressor (0.7731872017249168)
2. Random Forest Regressor (0.7597884089647924)
3. Multi-Layer Perceptron Regressor (0.7520022975479691)

## Hyper-parameter Tuning

In [ ]:
# Parameter grid for K-Neighbors Regressor
param_grid_KNreg = {
    "n_neighbors": list(range(1, 31)),
    "weights": ["uniform", "distance"],
    "p": [1, 2],                         # Metric: 1 for Manhattan, 2 for Euclidean
}

# Parameter grid for Random Forest Regressor
param_grid_rf = {
    "n_estimators": [int(x) for x in range(50, 301, 25)],
    "max_depth": [None] + list(range(5, 51, 5)),
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4, 6],
    "max_features": ["sqrt", "log2", None],
}

# Parameter grid for Multi-Layer Perceptron Regressor
param_grid_mlpreg = {
    "hidden_layer_sizes": [(50,), (100,), (50, 50)],
    "activation": ["relu"],
    "solver": ["adam"],
    "alpha": [0.0001, 0.01],
    "learning_rate": ["adaptive"],
    "max_iter": [1500]
}

## Model fitting (hyper-parameter tuning)

In [ ]:
# Tuning K-Neighbors Regressor
new_KNreg = GridSearchCV(KNreg, param_grid_KNreg, cv=5, scoring='r2', n_jobs=-1)
new_KNreg.fit(X_train_transformed, y_train)

In [ ]:
# Best model of K-Neighbors Regressor
new_KNreg.best_params_

The best parameters of **K-Neighbors Regressor** are:

- n_neighbors = 10
- p = 2
- weights = "distance"

In [ ]:
# Tuning Random Forest Regressor
new_rf = RandomizedSearchCV(rf, param_grid_rf, n_iter=50, cv=5, scoring='r2', n_jobs=-1, random_state=42)
new_rf.fit(X_train_transformed, y_train)

In [ ]:
# Best model of Random Forest Regressor
new_rf.best_params_

The best parameters of **Random Forest Regressor** are:

- n_estimators = 125
- min_samples_split = 15
- min_samples_leaf = 1
- max_features = None
- max_depth = 30

In [ ]:
# Tuning Multi-Layer Perceptron Regressor
new_mlpreg = RandomizedSearchCV(mlpreg, param_grid_mlpreg, n_iter=20, cv=5, scoring='r2', n_jobs=-1, random_state=42)
new_mlpreg.fit(X_train_transformed, y_train)

In [ ]:
# Best model of Multi-Layer Perceptron Regressor
new_mlpreg.best_params_

The best parameters of **Multi-Layer Perceptron Regressor** are:

- solver = "adam"
- max_iter = 1500
- learning_rate = "adaptive"
- hidden_layer_sizes = (100,)
- alpha = 0.01
- activation = "relu"

## Model performance evaluation (tuned models)

In [ ]:
new_KNreg_r2 = r2_score(y_val, new_KNreg.predict(X_val_transformed))
new_KNreg_r2

In [ ]:
new_rf_r2 = r2_score(y_val, new_rf.predict(X_val_transformed))
new_rf_r2

In [ ]:
new_mlpreg_r2 = r2_score(y_val, new_mlpreg.predict(X_val_transformed))
new_mlpreg_r2

In [ ]:
# Tuned models r2_score Comparision Bar Plot
models = ["K-Neighbors", "Random Forest", "Multi-Layer Perceptron"]
scores = [new_KNreg_r2, new_rf_r2, new_mlpreg_r2]

sns.barplot(x=models, y=scores, palette="tab10")
plt.xlabel("Models")
plt.ylabel("R2-Score")
plt.title("Model-wise R2-Scores on Validation Set")
plt.ylim(0.7, 0.8)
plt.show()

So based on the hyper-parameter tuning, we can observe that in our case, the **Tuned Multi-Layer Perceptron Regressor** is giving the best r2-score on the validation set.

## Voting and Stacking Regressors

In [ ]:
new_KNreg = KNeighborsRegressor(**new_KNreg.best_params_)
new_rf = RandomForestRegressor(**new_rf.best_params_)
new_mlpreg = MLPRegressor(**new_mlpreg.best_params_)

In [ ]:
vote = VotingRegressor([
    ("K-Neighbors", new_KNreg),
    ("Random Forest", new_rf),
    ("Multi-Layer Perceptron", new_mlpreg),
], n_jobs=-1)
vote.fit(X_train_transformed, y_train)

In [ ]:
vote.score(X_val_transformed, y_val)

In [ ]:
stack = StackingRegressor([
    ("K-Neighbors", new_KNreg),
    ("Random Forest", new_rf),
    ("Multi-Layer Perceptron", new_mlpreg),
], final_estimator=LinearRegression(), cv=5, n_jobs=-1)
stack.fit(X_train_transformed, y_train)

In [ ]:
stack.score(X_val_transformed, y_val)

Among the two, **Voting Regressor** is performing better and will be used for the final model submission.

# Test dataset processing

In [ ]:
# Dropping the id column
test_df_cleaned = test_df.drop("id", axis=1)

# Handling size column
test_df_cleaned["size"] = test_df_cleaned["size"].apply(extract_bhk_number)

# Handling availability column
ready_to_move_mask = test_df_cleaned["availability"] == "Ready To Move"
test_df_cleaned["availability"] = np.where(ready_to_move_mask, 1, 0)

# Handling location column
location_stats = test_df_cleaned["location"].value_counts()
locations_to_group = location_stats[location_stats <= 10].index
test_df_cleaned["location"] = test_df_cleaned["location"].apply(lambda x: "Other" if x in locations_to_group else x)

# Transforming the test set
X_test_transformed = pipeline.transform(test_df_cleaned)
X_test_transformed.shape

In [ ]:
# Having a look at the transformed test set
X_test_transformed.head()

In [ ]:
# Submission CSV generation
y_pred = vote.predict(X_test_transformed)

submission = pd.DataFrame({
    "id": range(0, X_test_transformed.shape[0]),
    "target": y_pred,
})
submission.to_csv("submission.csv", index=False)